# Crear variables

Las columnas de una tabla son lo que alguien registró, no necesariamente lo que necesita un modelo. Una variable calculada a partir de las demás puede expresar directamente lo que un modelo tendría que descubrir por sí mismo, o no podría descubrir nunca: habitaciones por hogar en lugar del total de habitaciones y el total de hogares de un distrito, la antigüedad de un coche en lugar de la fecha en que se matriculó. Crear este tipo de variables, normalmente a partir del conocimiento del dominio, se llama **ingeniería de variables** (*feature engineering*).

## Cocientes

Los distritos de California de [Valores atípicos y valores topados](outliers.ipynb). Las variables nuevas se prueban solo con las filas de entrenamiento, ya que elegirlas es una decisión sobre el modelo:

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

housing_csv = Path("data/housing/housing.csv")
if not housing_csv.exists():
    housing_csv.parent.mkdir(parents=True, exist_ok=True)
    archive, _ = urllib.request.urlretrieve("https://github.com/ageron/data/raw/main/housing.tgz")
    with tarfile.open(archive) as tar:
        tar.extractall(path="data", filter="data")
housing = pd.read_csv(housing_csv)
train, test = train_test_split(housing, test_size=0.2, random_state=42)
train.head(3)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
14196,-117.22,32.75,34.0,6001.0,1111.0,2654.0,1072.0,4.5878,291000.0,NEAR OCEAN
8267,-117.03,32.69,10.0,901.0,163.0,698.0,167.0,4.6648,156100.0,NEAR OCEAN
17445,-122.27,37.74,28.0,6909.0,1554.0,2974.0,1484.0,3.6875,353900.0,NEAR BAY


Los totales (habitaciones, dormitorios, personas, hogares) miden sobre todo lo grande que es un distrito, lo cual dice poco de sus viviendas: un distrito de 3000 habitaciones puede ser 1000 pisos pequeños o 300 casas grandes. Los cocientes describen la vivienda típica:

In [ ]:
import numpy as np
def add_ratios(frame):
    return frame.assign(
        rooms_per_household=frame["total_rooms"] / frame["households"],
        bedrooms_ratio=frame["total_bedrooms"] / frame["total_rooms"],
        people_per_household=frame["population"] / frame["households"],
    )


with_ratios = add_ratios(train)
with_ratios.corr(numeric_only=True)["median_house_value"].drop("median_house_value").sort_values(
    key=np.abs, ascending=False).round(3)

median_income           0.687
bedrooms_ratio         -0.255
rooms_per_household     0.147
latitude               -0.146
total_rooms             0.134
housing_median_age      0.105
households              0.066
total_bedrooms          0.050
longitude              -0.044
population             -0.026
people_per_household   -0.022
Name: median_house_value, dtype: float64

La proporción de habitaciones que son dormitorios tiene más correlación con el valor que cualquier total (−0,26: los distritos donde menos habitaciones son dormitorios, es decir, viviendas con más espacio de estar, valen más), y también las habitaciones por hogar (0,15). Las personas por hogar tienen una correlación cercana a 0, pero [Distribuciones asimétricas](skewed_distributions.ipynb) mostró una relación real oculta por sus valores extremos: una correlación es un primer vistazo, no un veredicto ([La correlación y lo que se le escapa](correlation.ipynb)).

La prueba de verdad es un modelo. Los cocientes se calculan dentro del pipeline con un `FunctionTransformer`, que aplica una función a todo lo que pasa por él, aquí la `add_ratios` de arriba. Una regresión lineal y un bosque aleatorio, con las variables numéricas originales y con los tres cocientes añadidos:

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

numerical = ["longitude", "latitude", "housing_median_age", "total_rooms", "total_bedrooms", "population",
             "households", "median_income"]
models = {
    "linear regression": LinearRegression(),
    "random forest": RandomForestRegressor(n_estimators=50, random_state=0, n_jobs=-1),
}
preprocessing = {
    "original features": [SimpleImputer(strategy="median"), StandardScaler()],
    "with the ratios": [FunctionTransformer(add_ratios), SimpleImputer(strategy="median"), StandardScaler()],
}
scores = pd.DataFrame({
    (features_name, model_name): cross_val_score(
        make_pipeline(*steps, model), train[numerical], train["median_house_value"], cv=5)
    for features_name, steps in preprocessing.items()
    for model_name, model in models.items()
})
pd.DataFrame({"mean R²": scores.mean(), "standard deviation across folds": scores.std()}).round(3)

mean R²  standard deviation across folds
original features linear regression    0.632                            0.014
                  random forest        0.811                            0.003
with the ratios   linear regression    0.641                            0.010
                  random forest        0.800                            0.007

Las dos versiones se puntúan con los mismos cinco pliegues, así que el cambio dentro de cada pliegue es una comparación más justa que la dispersión entre pliegues, que refleja sobre todo lo difícil que es cada pliegue:

In [ ]:
change = scores["with the ratios"] - scores["original features"]
pd.DataFrame({"mean change in R²": change.mean(), "folds where the ratios helped": (change > 0).sum()}).round(3)

,mean change in R²,folds where the ratios helped
linear regression,0.008,4
random forest,-0.011,0


La regresión lineal gana algo menos de un punto de R², y en cuatro de los cinco pliegues: solo puede sumar variables ponderadas, así que un cociente es algo que no podría construir por sí misma. El bosque aleatorio no gana; pierde alrededor de un punto, en todos los pliegues. Los árboles combinan variables mediante divisiones sucesivas, así que con la ubicación y los ingresos pueden aproximar buena parte de lo que añaden los cocientes; por qué las columnas extra le cuestan entonces un poco al bosque es algo que esta tabla no explica. Una variable que ayuda a un modelo no tiene por qué ayudar a otro, así que las variables se conservan o se descartan midiendo, como aquí.

## Fechas

Una fecha no se puede usar tal cual, pero contiene números que sí. Cuatro coches vendidos, con sus fechas de matriculación y de venta:

In [ ]:
sales = pd.DataFrame({
    "registered": pd.to_datetime(["2016-03-14", "2019-07-01", "2012-11-23", "2021-02-09"]),
    "sold": pd.to_datetime(["2024-12-20", "2025-01-05", "2025-06-15", "2025-07-01"]),
})
sales.assign(
    age_years=((sales["sold"] - sales["registered"]).dt.days / 365.25).round(1),
    sale_year=sales["sold"].dt.year,
    sale_month=sales["sold"].dt.month,
    sale_weekday=sales["sold"].dt.day_name(),
)

,registered,sold,age_years,sale_year,sale_month,sale_weekday
0,2016-03-14,2024-12-20,8.8,2024,12,Friday
1,2019-07-01,2025-01-05,5.5,2025,1,Sunday
2,2012-11-23,2025-06-15,12.6,2025,6,Sunday
3,2021-02-09,2025-07-01,4.4,2025,7,Tuesday


El **tiempo transcurrido** entre dos fechas (la antigüedad del coche al venderse) suele ser lo más informativo; el año, el mes y el día de la semana recogen tendencias y estaciones.

El mes tiene una trampa. Como número, diciembre es 12 y enero 1, a once de distancia, mientras que están tan cerca como junio y julio. Una **codificación cíclica** coloca los meses alrededor de un círculo, con un seno y un coseno, para que la distancia entre dos meses sea la misma en cualquier punto del año:

In [ ]:
month = sales["sold"].dt.month
cyclic = pd.DataFrame({"month_sin": np.sin(2 * np.pi * month / 12), "month_cos": np.cos(2 * np.pi * month / 12)})


def gap(table, first, second):
    return np.sqrt(((table.iloc[first] - table.iloc[second]) ** 2).sum())


pd.DataFrame({
    "month as a number": [gap(month.to_frame(), 0, 1), gap(month.to_frame(), 2, 3)],
    "month on a circle": [gap(cyclic, 0, 1), gap(cyclic, 2, 3)],
}, index=["December to January", "June to July"]).round(2)

,month as a number,month on a circle
December to January,11.0,0.52
June to July,1.0,0.52


En el círculo, los dos pares de meses consecutivos están a 0,52 de distancia. Lo mismo vale para las horas del día y los días de la semana. Codificar el mes con *one-hot* es la otra opción habitual ([Codificar categorías](../categorical_encoding.ipynb)): doce columnas sin ningún orden, en lugar de dos con el orden del círculo.

## Una variable dentro de un texto

[Tipos de variables](kinds_of_features.ipynb) vio que los nombres de los pasajeros del Titanic son inútiles tal cual, casi uno por pasajero, pero contienen un tratamiento. Una expresión regular toma la palabra entre la coma y el punto:

In [ ]:
from sklearn.datasets import fetch_openml

passengers = fetch_openml("titanic", version=1, as_frame=True, data_home="data").frame
passengers_train, passengers_test = train_test_split(passengers, test_size=0.2, stratify=passengers["survived"], random_state=42)
titles = passengers_train["name"].str.extract(r",\s*([^.]+)\.")[0].rename("title")
titles.value_counts()

title
Mr              606
Miss            210
Mrs             156
Master           51
Rev               6
Dr                6
Col               3
Mlle              2
Ms                2
Major             1
Don               1
the Countess      1
Capt              1
Dona              1
Name: count, dtype: int64

Cuatro tratamientos comunes y diez raros. Agruparlos por lo que significan es una regla fija, decidida a partir del conocimiento de la época, no de los datos: *Mlle* y *Ms* como *Miss*, *Mme* como *Mrs*, y el resto (clero, médicos, oficiales, nobleza) como *rare*. La tasa de supervivencia de cada grupo:

In [ ]:
title_groups = {"Mr": "Mr", "Mrs": "Mrs", "Mme": "Mrs", "Miss": "Miss", "Mlle": "Miss", "Ms": "Miss", "Master": "Master"}
title = titles.map(title_groups).fillna("rare").rename("title")
passengers_train["survived"].astype(int).groupby(title).agg(["mean", "size"]).round(2)

,mean,size
title,,
Master,0.49,51
Miss,0.67,214
Mr,0.17,606
Mrs,0.79,156
rare,0.30,20


*Master* era el tratamiento de los niños: sobrevivió el 49 % de ellos, frente al 17 % de los hombres llamados *Mr*. El tratamiento separa a los niños de los hombres, cosa que el sexo no puede, y lo hace para todos los pasajeros, mientras que la edad, que podría hacer lo mismo, falta en uno de cada cinco ([Valores ausentes](../missing_values.ipynb)).

El camarote, que falta en la mayoría de los pasajeros, también tiene una variable dentro: su primera letra es la cubierta:

In [ ]:
deck = passengers_train["cabin"].str[0].rename("deck")
passengers_train["survived"].astype(int).groupby(deck, dropna=False).agg(["mean", "size"]).round(2)

,mean,size
deck,,
A,0.50,16
B,0.69,48
C,0.62,73
D,0.68,41
E,0.74,31
F,0.67,12
G,0.67,3
T,0.00,1
NaN,0.31,822


En las cubiertas con más de un puñado de pasajeros, sobrevivió entre el 50 % y el 74 %, frente al 31 % de los pasajeros sin camarote registrado (`NaN`). La cubierta añade un poco a lo que ya se sabía por si el camarote estaba registrado, y solo para la quinta parte de los pasajeros que lo tienen.

## Intervalos y productos

Dos transformadores crean variables que permiten a un modelo lineal ajustar formas distintas de una línea recta:

- **Intervalos.** `KBinsDiscretizer` corta un número en intervalos y los codifica como categorías, para que un modelo lineal pueda dar a cada intervalo su propio peso: una función escalonada en lugar de una línea. Con `strategy="quantile"`, los bordes se eligen para que cada intervalo contenga más o menos el mismo número de filas de entrenamiento; se aprenden:

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

incomes = pd.DataFrame({"income": [18, 22, 25, 28, 31, 35, 60, 240]})
binner = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf")
binner.fit(incomes)
binner.bin_edges_[0]

array([ 18.,  25.,  35., 240.])

- **Productos.** `PolynomialFeatures` añade los cuadrados de las variables y sus productos dos a dos (con `degree=2`). Un producto de dos variables, una *interacción*, permite que el efecto de una dependa de la otra; un cuadrado permite que una línea se convierta en una curva. La parábola de [La correlación y lo que se le escapa](correlation.ipynb), que una línea recta no puede seguir en absoluto:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

parabola = pd.DataFrame({"x": np.linspace(-3, 3, 61)})
y_parabola = parabola["x"] ** 2
pd.Series({
    "x": LinearRegression().fit(parabola, y_parabola).score(parabola, y_parabola),
    "x and x²": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(parabola, y_parabola).score(parabola, y_parabola),
}).round(3).rename("R² of a linear regression")

x           0.0
x and x²    1.0
Name: R² of a linear regression, dtype: float64

Con $x^2$ como variable, la regresión lineal ajusta la parábola exactamente. El número de productos crece rápido con el número de variables (55 para diez variables con grado 2), y también el riesgo de sobreajuste, por eso las variables polinómicas suelen ir con un modelo regularizado ([Regularización](../../supervised-learning/ml-fundamentals/regularization.ipynb)).

## Calculadas a partir de cada fila, o aprendidas

Cada variable de este notebook pertenece a uno de dos tipos, y la diferencia decide dónde se calcula:

| Tipo | Ejemplos | Dónde |
| --- | --- | --- |
| Calculada solo a partir de cada fila, con una regla fija | Cocientes, el tiempo transcurrido entre dos fechas, el mes como seno y coseno, un tratamiento extraído de un nombre y agrupado por significado, un producto de dos columnas | En cualquier sitio: una vez, al construir la tabla, para todos los que la usen, o en el pipeline del modelo |
| Aprendida de los datos | Intervalos con bordes tomados de los datos, el objetivo medio de cada categoría ([Codificar categorías](../categorical_encoding.ipynb)), la similitud con clústeres de ubicaciones (Aprendizaje no supervisado *(próximamente)*), cualquier cosa calculada a partir de otras filas | Solo en el pipeline del modelo, ajustada con las filas de entrenamiento |

A menudo una variable de regla fija se pone igualmente en el pipeline, como se hizo aquí con `add_ratios`, para que el pipeline pueda recibir una fila en bruto en producción y calcularlo todo por sí mismo ([Viviendas de California con pipelines](../../supervised-learning/basic-workflows/california_with_pipelines.ipynb)). Una variable aprendida no tiene elección: si se calcula con todas las filas, deja que las filas de prueba moldeen los datos de entrenamiento. [Dónde termina la preparación de datos](../where_preparation_ends.ipynb) traza esa línea para cada paso de este tema.